<a href="https://colab.research.google.com/github/jarekwan/praca_inzynierska/blob/main/choose_source.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/ml_project', exist_ok=True)

print("OK — drive mounted and directory exists")



Mounted at /content/drive
OK — drive zamontowany i katalog istnieje


In [ ]:
%%writefile /content/drive/MyDrive/ml_project/choose_source.py
# -*- coding: utf-8 -*-
import os
import shutil
import pandas as pd
import requests
import ipywidgets as widgets
from IPython.display import display
from google.colab import drive

# mount google drive and prepare target directory
drive.mount('/content/drive', force_remount=True)
TARGET_DIR = "/content/drive/MyDrive/ml_project"
os.makedirs(TARGET_DIR, exist_ok=True)


def load_data(source: str, path: str) -> pd.DataFrame:
    if source == "csv":
        return pd.read_csv(path)
    elif source == "json":
        return pd.read_json(path)
    elif source == "api":
        r = requests.get(path)
        r.raise_for_status()
        data = r.json()
        return pd.DataFrame(data)
    else:
        raise ValueError("unknown data source type")


def choose_source():

    fmt = widgets.Dropdown(
        options=[('csv', 'csv'), ('json', 'json'), ('api', 'api')],
        description='Format:'
    )

    src = widgets.Dropdown(
        options=[('File path', 'path'), ('Upload file', 'upload')],
        description='Source:'
    )

    file_path = widgets.Text(description='Path:', placeholder='Enter file path...')
    api_url = widgets.Text(description='URL:', placeholder='Enter API URL...')
    upload_btn = widgets.FileUpload(description='Upload', multiple=False)
    ok_btn = widgets.Button(description='Confirm')
    out = widgets.Output()

    src.layout.display = 'none'
    file_path.layout.display = 'none'
    upload_btn.layout.display = 'none'
    api_url.layout.display = 'none'


    def fmt_change(change):
        src.layout.display = 'none'
        file_path.layout.display = 'none'
        upload_btn.layout.display = 'none'
        api_url.layout.display = 'none'

        if change['new'] in ['csv', 'json']:
            src.layout.display = 'block'
        else:
            api_url.layout.display = 'block'


    def src_change(change):
        file_path.layout.display = 'none'
        upload_btn.layout.display = 'none'

        if change['new'] == 'path':
            file_path.layout.display = 'block'
        else:
            upload_btn.layout.display = 'block'


    fmt.observe(fmt_change, names='value')
    src.observe(src_change, names='value')


    def on_ok(b):
        out.clear_output()
        format_selected = fmt.value

        try:
            if format_selected == 'api':
                url = api_url.value.strip()
                if not url:
                    raise ValueError('API URL is required')

                df = load_data('api', url)

                with out:
                    print('Data loaded from API.')
                    display(df.head())
                return

            ext = '.csv' if format_selected == 'csv' else '.json'
            final_path = os.path.join(TARGET_DIR, 'upload' + ext)

            for old_ext in ['.csv', '.json']:
                p = os.path.join(TARGET_DIR, 'upload' + old_ext)
                if os.path.exists(p):
                    os.remove(p)

            if src.value == 'path':
                p = file_path.value.strip()
                if not os.path.exists(p):
                    raise FileNotFoundError('File not found')
                shutil.copy(p, final_path)

            else:
                if not upload_btn.value:
                    raise ValueError('No file uploaded')
                file = list(upload_btn.value.values())[0]
                with open(final_path, 'wb') as f:
                    f.write(file['content'])

            df = load_data(format_selected, final_path)

            with out:
                print('File copied and data loaded.')
                print('Saved as:', final_path)
                display(df.head())

        except Exception as e:
            with out:
                print('Error:', e)

    ok_btn.on_click(on_ok)

    display(widgets.VBox([
        fmt, src, file_path, api_url, upload_btn, ok_btn, out
    ]))


#choose_source()


Writing /content/drive/MyDrive/ml_project/choose_source.py
